# Preparación, limpieza y estructuración de los datos del proyecto

**Aprendiz:** Juan Camilo Montes Velásquez
**Programa:** Tecnólogo en Análisis y Desarrollo de Software, SENA
**Evidencia:** GA1-220501094-01-AA1-EV07 – Preparación, limpieza y estructuración de datos del proyecto individual con Pandas

**Proyecto:** ¿Qué gana una partida de League of Legends?

**Pregunta analítica:** ¿Qué variables de los primeros 15 minutos de una partida (diferencia de oro, torres y objetivos neutrales) influyen más en la probabilidad de ganar en partidas profesionales de League of Legends, y se mantienen en partidas ranked de alto nivel?

**Pregunta orientadora:** ¿Cómo preparar y estructurar los datos del proyecto individual para obtener información confiable que permita responder la pregunta analítica planteada?

Este cuaderno aplica las técnicas de las lecciones 2, 3 y 4 del material *Preparar y estructurar datos con Python* a los dos datasets del proyecto. Parte de los problemas encontrados en el diagnóstico de calidad (`notebooks/data_quality.ipynb` y `reports/data_quality.md`). Debe ejecutarse de principio a fin (*Run → Run All Cells*), porque cada sección usa las variables de la anterior.

## Preparación del entorno y carga de los datos originales

In [1]:
import numpy as np
import pandas as pd
from pathlib import Path

RAW = Path('../data/raw')
PROCESSED = Path('../data/processed')
PROCESSED.mkdir(exist_ok=True)

pd.set_option('display.max_columns', 40)
print('pandas', pd.__version__)

pandas 3.1.1


In [2]:
pro_raw = pd.read_csv(RAW / '2025_LoL_esports_match_data_from_OraclesElixir.csv', low_memory=False)
ranked_raw = pd.read_csv(RAW / 'high_diamond_ranked_10min.csv')

print('Profesional (Oracle\'s Elixir):', pro_raw.shape)
print('Ranked Diamante (Kaggle):      ', ranked_raw.shape)

Profesional (Oracle's Elixir): (120528, 165)
Ranked Diamante (Kaggle):       (9879, 40)


Los archivos de `data/raw/` **no se modifican**: todas las transformaciones se hacen sobre copias en memoria y los resultados se guardan en `data/processed/`.

### Problemas del diagnóstico que guían la limpieza

| # | Dataset | Problema (diagnóstico EV05) | Acción propuesta |
|---|---|---|---|
| 1 | Profesional | 12 filas por partida (10 jugadores + 2 equipos) | Trabajar con las filas de equipo |
| 2 | Profesional | Nulos en las variables a 10 y 15 minutos (partidas `partial`, casi todas de la LPL) | Marcar y excluir del análisis a 15 minutos |
| 3 | Profesional | `patch` decimal y `date` como texto | Convertir a texto con dos decimales y a fecha |
| 4 | Profesional | `year` no coincide con `date` | Usar `date` para el periodo y tratar `year` como temporada |
| 5 | Profesional | Partidas sin primer objetivo | Revisarlas y decidir |
| 6 | Ranked | Valores atípicos en wards | Revisar y decidir |
| 7 | Ranked | Columnas redundantes | Dejar una columna por variable |
| 8 | Ambos | Estructuras distintas (fila por equipo vs. fila por partida) | Llevarlos a la misma forma para compararlos |

# Lección 2: Limpieza y estandarización de los datos

Orden de trabajo, igual que en el material: estado inicial, duplicados, unidad de análisis, nulos, inconsistencias, tipos y texto. Después todo se empaqueta en funciones, se encadena en un pipeline con `pipe()` y se valida con `assert`.

### 2.1 Estado inicial de los datos

In [3]:
def perfil_calidad(df, nombre):
    """Resume en una fila la calidad de una tabla."""
    return {
        'dataset': nombre,
        'filas': len(df),
        'columnas': df.shape[1],
        'nulos': int(df.isna().sum().sum()),
        'columnas_con_nulos': int((df.isna().sum() > 0).sum()),
        'duplicados': int(df.duplicated().sum()),
        'memoria_mb': round(df.memory_usage(deep=True).sum() / 1e6, 1),
    }

calidad_inicial = pd.DataFrame([perfil_calidad(pro_raw, 'pro_raw'),
                                perfil_calidad(ranked_raw, 'ranked_raw')])
calidad_inicial

,dataset,filas,columnas,nulos,columnas_con_nulos,duplicados,memoria_mb
0,pro_raw,120528,165,4976240,129,0,182.4
1,ranked_raw,9879,40,0,0,0,3.2


### 2.2 Registros duplicados

In [4]:
print('Profesional - filas idénticas:', pro_raw.duplicated().sum())
print('Profesional - gameid + participantid repetidos:', pro_raw.duplicated(['gameid', 'participantid']).sum())
print('Ranked - filas idénticas:', ranked_raw.duplicated().sum())
print('Ranked - gameId repetidos:', ranked_raw['gameId'].duplicated().sum())

Profesional - filas idénticas: 0
Profesional - gameid + participantid repetidos: 0
Ranked - filas idénticas: 0
Ranked - gameId repetidos: 0


**Interpretación:** ninguno de los dos datasets tiene filas duplicadas ni claves repetidas. En el profesional, la clave de una fila es `gameid` + `participantid`; en el ranked, `gameId`. **Decisión:** no hay nada que eliminar, pero la unicidad de las claves se vuelve a comprobar con `assert` en la validación y en la integridad final.

### 2.3 Unidad de análisis: filas de equipo

In [5]:
print(pro_raw['position'].value_counts())
print()
print('Participantes de las filas de equipo:', sorted(pro_raw.loc[pro_raw['position'] == 'team', 'participantid'].unique()))

position
top     20088
jng     20088
mid     20088
bot     20088
sup     20088
team    20088
Name: count, dtype: int64

Participantes de las filas de equipo: [np.int64(100), np.int64(200)]


**Interpretación:** cada partida profesional ocupa 12 filas: 5 posiciones de jugador por equipo y 2 filas de equipo (`participantid` 100 y 200). Como la pregunta es qué hace ganar a un **equipo**, la unidad de análisis es la fila de equipo. Además, los objetivos (`firstdragon`, `firsttower`…) solo están en esas filas. **Decisión:** trabajar con las 20 088 filas de equipo, dos por partida.

### 2.4 Selección de columnas

Para la pregunta solo se necesitan las variables de **inicio de partida** y el resultado. Se descartan a propósito los totales de fin de partida (`dragons`, `towers`, `kills`, `totalgold`…): se conocen cuando la partida ya terminó, así que usarlos para explicar la victoria sería hacer trampa (*fuga de información*). Los nombres se pasan a inglés en formato `snake_case`.

In [6]:
COLUMNAS_PRO = {
    'gameid': 'game_id', 'datacompleteness': 'data_completeness', 'league': 'league',
    'year': 'season', 'split': 'split', 'playoffs': 'playoffs', 'date': 'date',
    'game': 'game_in_series', 'patch': 'patch', 'side': 'side', 'teamname': 'team',
    'gamelength': 'game_length_s', 'result': 'result',
    'firstblood': 'first_blood', 'firstdragon': 'first_dragon',
    'firstherald': 'first_herald', 'firsttower': 'first_tower', 'void_grubs': 'void_grubs',
    'golddiffat10': 'gold_diff_10', 'xpdiffat10': 'xp_diff_10', 'csdiffat10': 'cs_diff_10',
    'killsat10': 'kills_10', 'deathsat10': 'deaths_10',
    'golddiffat15': 'gold_diff_15', 'xpdiffat15': 'xp_diff_15', 'csdiffat15': 'cs_diff_15',
    'killsat15': 'kills_15', 'deathsat15': 'deaths_15',
}
EARLY_COLS = ['gold_diff_10', 'xp_diff_10', 'cs_diff_10', 'kills_10', 'deaths_10',
              'gold_diff_15', 'xp_diff_15', 'cs_diff_15', 'kills_15', 'deaths_15', 'first_herald']

print(len(COLUMNAS_PRO), 'de', pro_raw.shape[1], 'columnas seleccionadas')

28 de 165 columnas seleccionadas


### 2.5 Análisis de valores faltantes

In [7]:
equipos_raw = pro_raw[pro_raw['position'] == 'team']
nulos_equipo = equipos_raw[list(COLUMNAS_PRO)].isna().sum()
print(nulos_equipo[nulos_equipo > 0])
print()
print('Nulos de golddiffat15 según datacompleteness:')
print(equipos_raw.groupby('datacompleteness')['golddiffat15'].apply(lambda s: s.isna().sum()))

split           2724
firstherald     1640
golddiffat10    1640
xpdiffat10      1640
csdiffat10      1640
killsat10       1640
deathsat10      1640
golddiffat15    1640
xpdiffat15      1640
csdiffat15      1640
killsat15       1640
deathsat15      1640
dtype: int64

Nulos de golddiffat15 según datacompleteness:
datacompleteness
complete       0
partial     1640
Name: golddiffat15, dtype: int64


In [8]:
sin_split = equipos_raw[equipos_raw['split'].isna()]
print('Ligas/torneos sin split:', sin_split['league'].value_counts().head(8).to_dict())

Ligas/torneos sin split: {'LJL': 424, 'AM': 354, 'HW': 228, 'WLDs': 192, 'MSI': 160, 'CD': 150, 'PRMP': 146, 'KeSPA Cup': 112}


**Interpretación y decisiones:**

| Columna | Nulos (filas de equipo) | Causa | Decisión |
|---|---|---|---|
| Variables a 10 y 15 min y `firstherald` | 1 640 | Partidas `partial`: la fuente no registró los datos por minuto (casi todas de la LPL) | **Conservar** las filas, marcarlas con `has_early_stats = False` y excluirlas del análisis a 15 min. Rellenarlas inventaría datos. |
| `split` | 2 724 | Torneos que no se dividen en splits (Worlds, MSI, LJL, copas) | **Rellenar** con `'No split'`: no es un dato perdido, es que no aplica. |

Las demás columnas seleccionadas no tienen nulos.

### 2.6 Inconsistencias

In [9]:
por_partida = equipos_raw.groupby('gameid')
resumen = pd.DataFrame({
    'first_blood': por_partida['firstblood'].sum(),
    'kills': por_partida['kills'].sum(),
    'first_dragon': por_partida['firstdragon'].sum(),
    'dragons': por_partida['dragons'].sum(),
    'first_herald': por_partida['firstherald'].sum(min_count=1),
    'heralds': por_partida['heralds'].sum(),
})

print('Partidas sin first blood:')
print(resumen.loc[resumen['first_blood'] == 0, ['first_blood', 'kills']])
print()
print('Partidas sin primer dragón -> dragones tomados:', resumen.loc[resumen['first_dragon'] == 0, 'dragons'].unique())
print('Partidas sin primer heraldo -> heraldos tomados:', resumen.loc[resumen['first_herald'] == 0, 'heralds'].unique())

Partidas sin first blood:
                    first_blood  kills
gameid                                
11770-11770_game_4          0.0     38

Partidas sin primer dragón -> dragones tomados: [0.]
Partidas sin primer heraldo -> heraldos tomados: [0.]


In [10]:
fechas = pd.to_datetime(equipos_raw['date'])
print(pd.crosstab(equipos_raw['year'], fechas.dt.year, rownames=['year'], colnames=['año de date']))

año de date   2025
year              
2025         19592
2026           496


In [11]:
wards_altos = (ranked_raw['blueWardsPlaced'] > 100) | (ranked_raw['redWardsPlaced'] > 100)
print('Partidas ranked con más de 100 wards de un equipo en 10 minutos:', wards_altos.sum())
print(ranked_raw[['blueWardsPlaced', 'redWardsPlaced']].describe().loc[['50%', 'max']])

Partidas ranked con más de 100 wards de un equipo en 10 minutos: 216
     blueWardsPlaced  redWardsPlaced
50%             16.0            16.0
max            250.0           276.0


**Interpretación y decisiones:**
- **1 partida sin first blood pero con 38 asesinatos** (`11770-11770_game_4`): es imposible, así que el dato no se registró. **Decisión:** dejar `first_blood` como nulo en esa partida en vez de asumir que ningún equipo lo logró.
- **Partidas sin primer dragón o sin primer heraldo:** en todas, ningún equipo tomó ese objetivo (0 dragones o 0 heraldos). Es coherente. **Decisión:** conservar.
- **`year` = 2026 en 496 filas con fecha de 2025:** `year` es el año de la temporada del torneo, no el de la partida. **Decisión:** renombrarla a `season` y usar `date` para el periodo; la validación comprueba que todas las fechas sean de 2025.
- **Wards atípicos en ranked** (216 partidas con más de 100 wards de un equipo en 10 minutos; mediana 16): los wards no forman parte de la pregunta. **Decisión:** no usarlos y excluirlos del dataset estructurado, en lugar de recortar valores que no se van a analizar.

### 2.7 Tipos de dato y formatos

In [12]:
print(pro_raw[['gameid', 'patch', 'date', 'firstblood', 'firstdragon', 'golddiffat15', 'result']].dtypes)
print()
print('patch como decimal:', sorted(pro_raw['patch'].unique())[8:11])
print('gameId ranked:', ranked_raw['gameId'].dtype)

gameid              str
patch           float64
date                str
firstblood      float64
firstdragon     float64
golddiffat15    float64
result            int64
dtype: object

patch como decimal: [np.float64(15.09), np.float64(15.1), np.float64(15.11)]
gameId ranked: int64


**Interpretación:** `patch` es decimal, así que el parche 15.10 aparece como `15.1`; `date` es texto; las banderas de primer objetivo y las diferencias por minuto son `float64` aunque sean enteros (por sus nulos), y el `gameId` del ranked es un número aunque sea un identificador. **Decisión:** `patch` a texto con dos decimales (`'15.10'`), `date` a fecha, banderas a `Int8` y diferencias a `Int16`/`Int32` (enteros *nullable* que admiten nulos), texto repetido (`league`, `split`, `side`) a `category`, `gameId` a texto y la duración de segundos a minutos.

### 2.8 Texto y categorías

In [13]:
for col in ['league', 'teamname', 'split', 'side', 'datacompleteness']:
    s = equipos_raw[col].dropna()
    print(f'{col:17} distintos={s.nunique():4}  espacios_extremos={(s != s.str.strip()).sum()}  '
          f'espacios_dobles={s.str.contains("  ", regex=False).sum()}  '
          f'variantes_por_mayusculas={s.nunique() - s.str.lower().nunique()}')
print()
print('Valores de side:', equipos_raw['side'].unique().tolist())

league            distintos=  45  espacios_extremos=0  espacios_dobles=0  variantes_por_mayusculas=0


teamname          distintos= 444  espacios_extremos=0  espacios_dobles=0  variantes_por_mayusculas=0
split             distintos=  20  espacios_extremos=0  espacios_dobles=0  variantes_por_mayusculas=0
side              distintos=   2  espacios_extremos=0  espacios_dobles=0  variantes_por_mayusculas=0
datacompleteness  distintos=   2  espacios_extremos=0  espacios_dobles=0  variantes_por_mayusculas=0

Valores de side: ['Blue', 'Red']


**Interpretación:** los textos ya vienen limpios: no hay espacios al inicio o al final, espacios dobles ni variantes del mismo nombre por mayúsculas. **Decisión:** igual se aplica `strip()` en la función de limpieza (por si llegan datos nuevos), se pasa `side` a minúsculas (`blue`/`red`) para que coincida con el esquema del dataset ranked y se convierten a categoría las columnas con pocos valores distintos.

### 2.9 Funciones reutilizables de limpieza

Cada función hace una sola cosa, recibe un DataFrame y devuelve otro. Así se pueden encadenar con `pipe()`.

In [14]:
registro = []

def anotar(df, dataset, paso):
    """Guarda una foto del estado de la tabla sin modificarla."""
    registro.append({'dataset': dataset, 'paso': paso, 'filas': len(df), 'columnas': df.shape[1],
                     'nulos': int(df.isna().sum().sum()),
                     'memoria_mb': round(df.memory_usage(deep=True).sum() / 1e6, 2)})
    return df

# ---------- Dataset profesional ----------
def filas_de_equipo(df):
    return df[df['position'] == 'team'].reset_index(drop=True)

def seleccionar_columnas(df, columnas):
    return df[list(columnas)].rename(columns=columnas)

def estandarizar_texto(df):
    df = df.copy()
    for col in ['league', 'team', 'split', 'data_completeness']:
        df[col] = df[col].str.strip()
    df['side'] = df['side'].str.strip().str.lower()
    df['split'] = df['split'].fillna('No split')
    return df

def corregir_tipos_pro(df):
    df = df.copy()
    df['patch'] = df['patch'].map(lambda p: f'{p:.2f}')
    df['date'] = pd.to_datetime(df['date'])
    df['game_length_min'] = (df.pop('game_length_s') / 60).round(2)
    df = df.astype({'season': 'int16', 'playoffs': 'int8', 'game_in_series': 'int8', 'result': 'int8',
                    'void_grubs': 'int8',
                    'first_blood': 'Int8', 'first_dragon': 'Int8', 'first_herald': 'Int8', 'first_tower': 'Int8',
                    'gold_diff_10': 'Int32', 'xp_diff_10': 'Int32', 'cs_diff_10': 'Int16',
                    'kills_10': 'Int16', 'deaths_10': 'Int16',
                    'gold_diff_15': 'Int32', 'xp_diff_15': 'Int32', 'cs_diff_15': 'Int16',
                    'kills_15': 'Int16', 'deaths_15': 'Int16',
                    'league': 'category', 'split': 'category', 'side': 'category',
                    'data_completeness': 'category'})
    return df

def corregir_first_blood(df):
    """Si ningún equipo tiene first blood pero hubo asesinatos, el dato no se registró: queda como nulo."""
    df = df.copy()
    total = df.groupby('game_id')['first_blood'].transform('sum')
    df.loc[total == 0, 'first_blood'] = pd.NA
    return df

def marcar_datos_tempranos(df):
    df = df.copy()
    df['has_early_stats'] = df[EARLY_COLS].notna().all(axis=1)
    return df

# ---------- Dataset ranked ----------
REDUNDANTES_RANKED = ['blueEliteMonsters', 'redEliteMonsters', 'blueCSPerMin', 'redCSPerMin',
                      'blueGoldPerMin', 'redGoldPerMin', 'redGoldDiff', 'redExperienceDiff']
FUERA_DE_ALCANCE_RANKED = ['blueWardsPlaced', 'redWardsPlaced', 'blueWardsDestroyed', 'redWardsDestroyed',
                           'blueAssists', 'redAssists', 'blueAvgLevel', 'redAvgLevel',
                           'blueTotalGold', 'redTotalGold', 'blueTotalExperience', 'redTotalExperience']

def quitar_columnas(df, columnas):
    return df.drop(columns=columnas)

def corregir_tipos_ranked(df):
    df = df.copy()
    df['gameId'] = df['gameId'].astype('str')
    enteros = df.columns.drop('gameId')
    df[enteros] = df[enteros].astype('int16')
    return df

### 2.10 Pipeline de limpieza con `pipe()`

In [15]:
registro = []

pro_clean = (pro_raw
    .pipe(anotar, 'profesional', '0 crudo')
    .pipe(filas_de_equipo)
    .pipe(anotar, 'profesional', '1 filas de equipo')
    .pipe(seleccionar_columnas, COLUMNAS_PRO)
    .pipe(anotar, 'profesional', '2 columnas de inicio de partida')
    .pipe(estandarizar_texto)
    .pipe(anotar, 'profesional', '3 texto y split estandarizados')
    .pipe(corregir_tipos_pro)
    .pipe(anotar, 'profesional', '4 tipos corregidos')
    .pipe(corregir_first_blood)
    .pipe(anotar, 'profesional', '5 first blood inconsistente a nulo')
    .pipe(marcar_datos_tempranos)
    .pipe(anotar, 'profesional', '6 marca de datos tempranos'))

ranked_clean = (ranked_raw
    .pipe(anotar, 'ranked', '0 crudo')
    .pipe(quitar_columnas, REDUNDANTES_RANKED)
    .pipe(anotar, 'ranked', '1 sin columnas redundantes')
    .pipe(quitar_columnas, FUERA_DE_ALCANCE_RANKED)
    .pipe(anotar, 'ranked', '2 sin columnas fuera de alcance')
    .pipe(corregir_tipos_ranked)
    .pipe(anotar, 'ranked', '3 tipos corregidos'))

informe_limpieza = pd.DataFrame(registro)
informe_limpieza

,dataset,paso,filas,columnas,nulos,memoria_mb
0,profesional,0 crudo,120528,165,4976240,182.35
1,profesional,1 filas de equipo,20088,165,411939,30.04
2,profesional,2 columnas de inicio de partida,20088,28,20764,5.92
3,profesional,3 texto y split estandarizados,20088,28,18040,5.94
4,profesional,4 tipos corregidos,20088,28,18040,2.64
5,profesional,5 first blood inconsistente a nulo,20088,28,18042,2.64
6,profesional,6 marca de datos tempranos,20088,29,18042,2.66
7,ranked,0 crudo,9879,40,0,3.16
8,ranked,1 sin columnas redundantes,9879,32,0,2.53
9,ranked,2 sin columnas fuera de alcance,9879,20,0,1.58


**Interpretación del registro de cambios:**
- **Profesional:** quedarse con las filas de equipo pasa de 120 528 a 20 088 filas; seleccionar las 28 columnas de inicio de partida baja los nulos de 411 939 a 20 764. Rellenar `split` elimina 2 724 nulos, y corregir los tipos baja la memoria de 5,9 a 2,6 MB. El paso del first blood agrega 2 nulos a propósito (la partida inconsistente). Los 18 042 nulos que quedan son los de las partidas `partial`, marcadas con `has_early_stats`.
- **Ranked:** se quitan 8 columnas redundantes y 12 fuera del alcance de la pregunta; con tipos enteros pequeños la memoria pasa de 3,2 a 0,6 MB. No había nulos.

### 2.11 Validación de la limpieza

In [16]:
def validar_pro(df):
    partidas = df.groupby('game_id')
    assert (partidas.size() == 2).all(), 'cada partida debe tener 2 equipos'
    assert (partidas['side'].nunique() == 2).all(), 'cada partida debe tener un lado azul y uno rojo'
    assert (partidas['result'].sum() == 1).all(), 'cada partida debe tener un ganador'
    fb = partidas['first_blood'].sum(min_count=1).dropna()
    assert (fb == 1).all(), 'first blood debe ser de un solo equipo'
    tempranas = df[df['has_early_stats']]
    assert (tempranas.groupby('game_id')['gold_diff_15'].sum() == 0).all(), 'diferencias de oro no simétricas'
    assert tempranas[EARLY_COLS].notna().all().all(), 'nulos en partidas con datos tempranos'
    assert df['patch'].str.fullmatch(r'\d+\.\d{2}').all(), 'formato de patch'
    assert (df['date'].dt.year == 2025).all(), 'fechas fuera de 2025'
    assert df['split'].notna().all(), 'split con nulos'
    return 'Validación del dataset profesional superada'

def validar_ranked(df):
    assert df['gameId'].is_unique, 'gameId repetido'
    assert df['blueWins'].isin([0, 1]).all(), 'blueWins con valores raros'
    assert ((df['blueFirstBlood'] + df['redFirstBlood']) == 1).all(), 'first blood inconsistente'
    assert ((df['blueKills'] == df['redDeaths']) & (df['redKills'] == df['blueDeaths'])).all(), 'kills y deaths no cuadran'
    assert df.isna().sum().sum() == 0, 'nulos en ranked'
    return 'Validación del dataset ranked superada'

print(validar_pro(pro_clean))
print(validar_ranked(ranked_clean))

Validación del dataset profesional superada
Validación del dataset ranked superada


In [17]:
print(pro_clean.dtypes)
pro_clean.head()

game_id                         str
data_completeness          category
league                     category
season                        int16
split                      category
playoffs                       int8
date                 datetime64[us]
game_in_series                 int8
patch                           str
side                       category
team                            str
result                         int8
first_blood                    Int8
first_dragon                   Int8
first_herald                   Int8
first_tower                    Int8
void_grubs                     int8
gold_diff_10                  Int32
xp_diff_10                    Int32
cs_diff_10                    Int16
kills_10                      Int16
deaths_10                     Int16
gold_diff_15                  Int32
xp_diff_15                    Int32
cs_diff_15                    Int16
kills_15                      Int16
deaths_15                     Int16
game_length_min             

,game_id,data_completeness,league,season,split,playoffs,date,game_in_series,patch,side,team,result,first_blood,first_dragon,first_herald,first_tower,void_grubs,gold_diff_10,xp_diff_10,cs_diff_10,kills_10,deaths_10,gold_diff_15,xp_diff_15,cs_diff_15,kills_15,deaths_15,game_length_min,has_early_stats
0,LOLTMNT03_179647,complete,LFL2,2025,Winter,0,2025-01-11 11:11:24,1,15.01,blue,IziDream,0,0,0,0,0,0,-1722,-662,-32,0,1,-3837,-469,-16,0,3,26.53,True
1,LOLTMNT03_179647,complete,LFL2,2025,Winter,0,2025-01-11 11:11:24,1,15.01,red,Team Valiant,1,1,1,1,1,6,1722,662,32,1,0,3837,469,16,3,0,26.53,True
2,LOLTMNT06_96134,complete,LFL2,2025,Winter,0,2025-01-11 12:06:37,1,15.01,blue,Esprit Shōnen,1,1,0,1,1,6,3104,1069,33,8,4,5069,2014,64,10,6,32.03,True
3,LOLTMNT06_96134,complete,LFL2,2025,Winter,0,2025-01-11 12:06:37,1,15.01,red,Skillcamp,0,0,1,0,0,0,-3104,-1069,-33,3,8,-5069,-2014,-64,5,10,32.03,True
4,LOLTMNT06_95160,complete,LFL2,2025,Winter,0,2025-01-11 13:07:47,1,15.01,blue,Karmine Corp Blue Stars,0,0,0,0,0,2,-915,-354,-30,5,5,118,1990,-43,10,6,29.70,True


### 2.12 Calidad antes y después de la limpieza

In [18]:
calidad_limpieza = pd.DataFrame([
    perfil_calidad(pro_raw, 'pro_raw (antes)'),
    perfil_calidad(pro_clean, 'pro_clean (después)'),
    perfil_calidad(ranked_raw, 'ranked_raw (antes)'),
    perfil_calidad(ranked_clean, 'ranked_clean (después)'),
])
calidad_limpieza

,dataset,filas,columnas,nulos,columnas_con_nulos,duplicados,memoria_mb
0,pro_raw (antes),120528,165,4976240,129,0,182.4
1,pro_clean (después),20088,29,18042,12,0,2.7
2,ranked_raw (antes),9879,40,0,0,0,3.2
3,ranked_clean (después),9879,20,0,0,0,0.6


In [19]:
nulos_restantes = pro_clean.isna().sum()
print('Nulos que quedan en pro_clean:')
print(nulos_restantes[nulos_restantes > 0])
print()
print('Filas con datos tempranos completos:', pro_clean['has_early_stats'].sum(), 'de', len(pro_clean))

Nulos que quedan en pro_clean:
first_blood        2
first_herald    1640
gold_diff_10    1640
xp_diff_10      1640
cs_diff_10      1640
kills_10        1640
deaths_10       1640
gold_diff_15    1640
xp_diff_15      1640
cs_diff_15      1640
kills_15        1640
deaths_15       1640
dtype: int64

Filas con datos tempranos completos: 18448 de 20088


**Interpretación:** el dataset profesional pasa de 120 528 filas, 165 columnas y casi 5 millones de nulos en 129 columnas a 20 088 filas de equipo, 29 columnas y 18 042 nulos en 12 columnas. Todos los nulos que quedan tienen causa conocida y están marcados (las 1 640 filas de partidas `partial` y las 2 filas del first blood no registrado). Las 18 448 filas con `has_early_stats = True` no tienen ningún nulo. El ranked conserva sus 9 879 partidas sin nulos, con la mitad de columnas y una quinta parte de la memoria.

# Lección 3: Filtrado, ordenamiento y agrupación de datos

Cada pregunta se traduce a "métrica por grupo". La métrica principal es la **tasa de victoria** (la media de `result`, que vale 0 o 1). El análisis a 15 minutos solo usa las partidas con datos tempranos completos.

In [20]:
pro15 = pro_clean[pro_clean['has_early_stats']].copy()
print('Equipos con datos a 15 minutos:', len(pro15), '| partidas:', pro15['game_id'].nunique())

Equipos con datos a 15 minutos: 18448 | partidas: 9224


### 3.1 Filtros con condiciones booleanas, `between`, `isin` y `query`

In [21]:
adelante_15 = pro15['gold_diff_15'] >= 1000
print('Equipos con 1 000 de oro o más de ventaja a los 15 min:', adelante_15.sum(),
      '| tasa de victoria:', round(pro15.loc[adelante_15, 'result'].mean(), 3))

parejas = pro15[pro15['gold_diff_15'].between(-1000, 1000)]
print('Equipos en partidas parejas (±1 000 de oro):', len(parejas),
      '| tasa de victoria:', round(parejas['result'].mean(), 3))

LIGAS_PRINCIPALES = ['LCK', 'LEC', 'LTA N', 'LTA S', 'LCP', 'LPL']
principales = pro15[pro15['league'].isin(LIGAS_PRINCIPALES)]
print('Equipos de ligas principales con datos a 15 min:', len(principales),
      principales['league'].value_counts().loc[lambda s: s > 0].to_dict())

remontadas = pro15.query('gold_diff_15 <= -3000 and result == 1')
print('Remontadas (ganaron yendo 3 000 o más abajo a los 15 min):', len(remontadas),
      'de', (pro15['gold_diff_15'] <= -3000).sum())

Equipos con 1 000 de oro o más de ventaja a los 15 min: 6121 | tasa de victoria: 0.809
Equipos en partidas parejas (±1 000 de oro): 6214 | tasa de victoria: 0.5
Equipos de ligas principales con datos a 15 min: 3180 {'LCK': 1110, 'LEC': 616, 'LCP': 582, 'LTA S': 444, 'LTA N': 428}
Remontadas (ganaron yendo 3 000 o más abajo a los 15 min): 125 de 2038


**Interpretación:** los equipos con 1 000 de oro o más de ventaja a los 15 minutos ganan el **80,9 %** de las veces, mientras que en las partidas parejas (±1 000) la tasa es exactamente **50 %**: la ventaja de oro marca la diferencia. Las remontadas existen, pero son raras: solo 125 de 2 038 equipos que iban 3 000 o más abajo terminaron ganando (6,1 %). En las ligas principales hay 3 180 equipos con datos a 15 minutos; la LPL no aparece porque todas sus partidas son `partial`.

### 3.2 Ordenamiento y selección de registros relevantes

In [22]:
cols = ['league', 'date', 'team', 'side', 'gold_diff_15', 'first_tower', 'game_length_min', 'result']
print('Las 5 mayores remontadas (menor diferencia de oro a los 15 min y aun así ganaron):')
print(pro15[pro15['result'] == 1].nsmallest(5, 'gold_diff_15')[cols].to_string())
print()
print('Las 5 mayores ventajas a los 15 min que terminaron en derrota:')
print(pro15[pro15['result'] == 0].nlargest(5, 'gold_diff_15')[cols].to_string())

Las 5 mayores remontadas (menor diferencia de oro a los 15 min y aun así ganaron):
      league                date                         team  side  gold_diff_15  first_tower  game_length_min  result
8018     LAS 2025-05-03 05:58:18  Hanwha Life Esports Academy  blue         -5777            0            35.78       1
12413     AM 2025-06-25 11:06:35                     wangting   red         -5565            0            30.23       1
18602     EM 2025-09-30 17:43:59                   Otter Side  blue         -5481            0            45.13       1
12831     AL 2025-07-16 17:13:47                 BAAM Esports   red         -5449            0            40.92       1
11985    VCS 2025-06-15 13:31:20          MGN Vikings Academy   red         -5407            0            34.72       1

Las 5 mayores ventajas a los 15 min que terminaron en derrota:
      league                date                                     team  side  gold_diff_15  first_tower  game_length_min  result
8

**Interpretación:** las cinco mayores remontadas y las cinco mayores ventajas perdidas son **las mismas cinco partidas vistas desde los dos equipos**: el que iba entre 5 400 y 5 800 de oro abajo a los 15 minutos ganó, y el rival que tenía esa ventaja perdió. Esto confirma que las diferencias de oro son simétricas. Ninguno de los equipos que remontaron tenía la primera torre, y todas esas partidas duraron más de 30 minutos: remontar exige tiempo.

### 3.3 Indicador 1: tasa de victoria según el primer objetivo

*Métrica por grupo:* tasa de victoria de los equipos que consiguieron cada primer objetivo frente a los que no.

In [23]:
def tasa_por_objetivo(df, objetivos, resultado):
    filas = []
    for nombre, columna in objetivos.items():
        g = df.groupby(columna, observed=True)[resultado].agg(['mean', 'size'])
        filas.append({'objetivo': nombre,
                      'tasa_victoria_con': round(g.loc[1, 'mean'], 3),
                      'tasa_victoria_sin': round(g.loc[0, 'mean'], 3),
                      'equipos_con': int(g.loc[1, 'size'])})
    tabla = pd.DataFrame(filas)
    tabla['diferencia_pp'] = ((tabla['tasa_victoria_con'] - tabla['tasa_victoria_sin']) * 100).round(1)
    return tabla.sort_values('tasa_victoria_con', ascending=False).reset_index(drop=True)

objetivos_pro = tasa_por_objetivo(pro15, {'Primera torre': 'first_tower', 'Primer dragón': 'first_dragon',
                                          'Primer heraldo': 'first_herald', 'First blood': 'first_blood'}, 'result')
objetivos_pro

,objetivo,tasa_victoria_con,tasa_victoria_sin,equipos_con,diferencia_pp
0,Primera torre,0.701,0.299,9224,40.2
1,Primer heraldo,0.665,0.336,9205,32.9
2,First blood,0.578,0.422,9224,15.6
3,Primer dragón,0.576,0.424,9218,15.2


In [24]:
# El dataset ranked tiene una fila por partida vista desde el equipo azul.
ranked_flags = ranked_clean.assign(
    dragon_10=(ranked_clean['blueDragons'] > 0).astype('int8'),
    heraldo_10=(ranked_clean['blueHeralds'] > 0).astype('int8'),
    torre_10=(ranked_clean['blueTowersDestroyed'] > 0).astype('int8'))

objetivos_ranked = tasa_por_objetivo(ranked_flags, {'Torre antes del min 10': 'torre_10',
                                                    'Dragón antes del min 10': 'dragon_10',
                                                    'Heraldo antes del min 10': 'heraldo_10',
                                                    'First blood': 'blueFirstBlood'}, 'blueWins')
objetivos_ranked

,objetivo,tasa_victoria_con,tasa_victoria_sin,equipos_con,diferencia_pp
0,Torre antes del min 10,0.754,0.486,464,26.8
1,Dragón antes del min 10,0.641,0.419,3576,22.2
2,First blood,0.599,0.397,4987,20.2
3,Heraldo antes del min 10,0.595,0.477,1857,11.8


**Interpretación (indicador 1):** en partidas profesionales, el objetivo temprano que más se relaciona con ganar es la **primera torre**: quien la consigue gana el **70,1 %** de las veces (40,2 puntos más que quien no). Le siguen el **primer heraldo** (66,5 %), el **first blood** (57,8 %) y el **primer dragón** (57,6 %). En ranked el orden es parecido: la **torre antes del minuto 10** da la tasa más alta (75,4 %, aunque solo 464 partidas), seguida del dragón (64,1 %) y el first blood (59,9 %). El heraldo antes del minuto 10 pesa menos en ranked (59,5 %).

Limitación: en el dataset profesional las banderas `first_*` indican quién tomó el objetivo primero, pero no en qué minuto; en ranked se sabe que ocurrió antes del minuto 10.

### 3.4 Indicador 2: tasa de victoria por tramo de diferencia de oro a los 15 minutos (categorización con `pd.cut`)

In [25]:
TRAMOS_ORO = [-np.inf, -3000, -1000, 1000, 3000, np.inf]
ETIQUETAS_ORO = ['atrás más de 3000', 'atrás 1000 a 3000', 'parejo (±1000)',
                 'adelante 1000 a 3000', 'adelante más de 3000']

pro15['gold_bracket_15'] = pd.cut(pro15['gold_diff_15'], bins=TRAMOS_ORO, labels=ETIQUETAS_ORO)

tasa_por_oro = (pro15.groupby('gold_bracket_15', observed=True)
                     .agg(equipos=('result', 'size'), tasa_victoria=('result', 'mean'))
                     .round(3))
tasa_por_oro['pct_equipos'] = (tasa_por_oro['equipos'] / tasa_por_oro['equipos'].sum() * 100).round(1)
tasa_por_oro

,equipos,tasa_victoria,pct_equipos
gold_bracket_15,,,
atrás más de 3000,2038,0.061,11.0
atrás 1000 a 3000,4083,0.255,22.1
parejo (±1000),6210,0.500,33.7
adelante 1000 a 3000,4081,0.745,22.1
adelante más de 3000,2036,0.939,11.0


**Interpretación (indicador 2):** la tasa de victoria sube casi en línea recta con la diferencia de oro a los 15 minutos: del **6,1 %** para los equipos más de 3 000 abajo al **93,9 %** para los que van más de 3 000 arriba. En el tramo parejo es 50 %. Una tercera parte de los equipos (33,7 %) llega al minuto 15 en una partida pareja, y en esos casos la partida se define después.

### 3.5 Indicador 3: diferencias promedio a los 15 minutos de ganadores y perdedores (`groupby` + `agg`)

In [26]:
ganadores_perdedores = (pro15
    .assign(kill_diff_15=pro15['kills_15'] - pro15['deaths_15'])
    .groupby('result')
    .agg(equipos=('team', 'size'),
         oro_medio=('gold_diff_15', 'mean'),
         oro_mediana=('gold_diff_15', 'median'),
         experiencia_media=('xp_diff_15', 'mean'),
         cs_medio=('cs_diff_15', 'mean'),
         kills_netos_medio=('kill_diff_15', 'mean'),
         larvas_medio=('void_grubs', 'mean'))
    .round(1)
    .rename(index={0: 'perdedores', 1: 'ganadores'}))
ganadores_perdedores

,equipos,oro_medio,oro_mediana,experiencia_media,cs_medio,kills_netos_medio,larvas_medio
result,,,,,,,
perdedores,9224,-1345.1,-1176.5,-951.5,-17.0,-1.6,1.8
ganadores,9224,1345.1,1176.5,951.5,17.0,1.6,2.4


**Interpretación (indicador 3):** a los 15 minutos, los ganadores van en promedio **1 345 de oro**, 952 de experiencia y 17 de CS por encima de su rival, con 1,6 asesinatos netos más, y toman más larvas del vacío (2,4 frente a 1,8). Los valores son simétricos (ganadores y perdedores tienen la misma cifra con signo opuesto) porque cada partida aporta un ganador y un perdedor con diferencias opuestas, lo que confirma que la limpieza dejó los datos coherentes.

### 3.6 Indicador 4: qué variable se relaciona más con ganar (correlación con el resultado)

La correlación va de -1 a 1: cuanto más lejos de 0, más fuerte es la relación entre la variable y ganar. Responde directamente a "qué variables influyen más". Correlación no es causalidad, pero sirve para ordenar las variables.

In [27]:
VARIABLES_15 = ['gold_diff_15', 'xp_diff_15', 'cs_diff_15', 'first_tower', 'first_dragon',
                'first_herald', 'first_blood', 'void_grubs']

correlacion_15 = (pro15[VARIABLES_15].astype('float')
                  .corrwith(pro15['result'].astype('float'))
                  .sort_values(ascending=False)
                  .round(3)
                  .rename('correlacion_con_ganar'))
correlacion_15.to_frame()

,correlacion_con_ganar
gold_diff_15,0.523
xp_diff_15,0.492
cs_diff_15,0.449
first_tower,0.402
first_herald,0.329
void_grubs,0.196
first_blood,0.155
first_dragon,0.152


**Interpretación (indicador 4):** ordenadas por su relación con ganar, las variables de los primeros 15 minutos quedan así: **diferencia de oro (0,523)**, diferencia de experiencia (0,492), diferencia de CS (0,449), **primera torre (0,402)**, primer heraldo (0,329), larvas del vacío (0,196), first blood (0,155) y primer dragón (0,152). Es decir: **la diferencia de recursos (oro y experiencia) es lo que más se relaciona con ganar**, y entre los objetivos la primera torre es el más importante. El primer dragón, que suele verse como clave, es el que menos pesa.

### 3.7 Tabla dinámica: tasa de victoria por liga según la primera torre

In [28]:
ligas_con_mas_partidas = (pro15.drop_duplicates('game_id')['league']
                          .value_counts().nlargest(8).index)

tabla_ligas = (pro15[pro15['league'].isin(ligas_con_mas_partidas)]
               .pivot_table(index='league', columns='first_tower', values='result',
                            aggfunc='mean', observed=True)
               .rename(columns={0: 'sin_primera_torre', 1: 'con_primera_torre'})
               .round(3)
               .sort_values('con_primera_torre', ascending=False))
tabla_ligas

first_tower,sin_primera_torre,con_primera_torre
league,,
LJL,0.211,0.789
PCS,0.265,0.735
LAS,0.287,0.713
LCKC,0.304,0.696
LCK,0.323,0.677
EM,0.342,0.658
LEC,0.347,0.653
LFL,0.368,0.632


In [29]:
print('Tasa de victoria por lado del mapa:')
print(pro15.groupby('side', observed=True)['result'].mean().round(3))

Tasa de victoria por lado del mapa:
side
blue    0.533
red     0.467
Name: result, dtype: float64


**Interpretación:** en las 8 ligas con más partidas, la primera torre siempre aumenta la probabilidad de ganar, pero el efecto cambia según la liga: en la LJL (Japón) quien la toma gana el 78,9 % de las veces, mientras que en la LFL (Francia) gana el 63,2 %. En la LCK y la LEC está entre 65 % y 68 %. Por lado del mapa, el **azul gana el 53,3 %** de las partidas: una ventaja pequeña que conviene tener en cuenta.

### 3.8 Relación de los resultados con la pregunta analítica

**Respuesta a la primera parte de la pregunta analítica:** en partidas profesionales de 2025, lo que más se relaciona con ganar es la **ventaja de recursos a los 15 minutos** (oro, luego experiencia y CS). Entre los objetivos tempranos pesan más la **primera torre** y el **primer heraldo** que el first blood o el primer dragón. Con 1 000 de oro de ventaja a los 15 minutos, un equipo gana 8 de cada 10 partidas.